In [1]:
questions = [
    {
        "question": "What expenses are covered under inpatient hospitalization?",
        "relevant_pages": [2, 3]
    },
    {
        "question": "What is the pre-hospitalization period?",
        "relevant_pages": [10]
    },
    {
        "question": "What is the post-hospitalization period?",
        "relevant_pages": [10]
    }
]

In [3]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_collection(
    name="insurance_documents"
)

print("ChromaDB connected successfully!")
print("Total chunks:", collection.count())

ChromaDB connected successfully!
Total chunks: 872


In [4]:
collection = chroma_client.get_collection(
    name="insurance_documents"
)

In [5]:
def retrieve_chunks(question, k=5):

    results = collection.query(
        query_texts=[question],
        n_results=k
    )

    return results

In [6]:
result = retrieve_chunks(
    "What is the pre-hospitalization period?",
    k=5
)

print(result["metadatas"][0])

[{'document_type': 'Prospectus / Sales Literature', 'source': 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'page': 2}, {'page': 2, 'source': 'care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'document_type': 'Prospectus / Sales Literature'}, {'document_type': 'Prospectus / Sales Literature', 'source': 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'page': 2}, {'document_type': 'Prospectus / Sales Literature', 'source': 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'page': 8}, {'source': 'care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf', 'page': 12, 'document_type': 'Prospectus / Sales Literature'}]


In [7]:
def get_retrieved_pages(question, k=5):

    results = collection.query(
        query_texts=[question],
        n_results=k
    )

    pages = []

    for metadata in results["metadatas"][0]:
        pages.append(metadata["page"])

    return pages

In [8]:
pages = get_retrieved_pages(
    "What is the pre-hospitalization period?",
    k=5
)

print(pages)

[2, 2, 2, 8, 12]


In [9]:
def precision_at_k(retrieved_pages, relevant_pages):

    relevant_count = sum(
        page in relevant_pages
        for page in retrieved_pages
    )

    return relevant_count / len(retrieved_pages)

In [10]:
retrieved = [2,2,2,8,12]
relevant = [2]

precision = precision_at_k(retrieved, relevant)

print("Precision@5:", precision)

Precision@5: 0.6


In [11]:
def recall_at_k(retrieved_pages, relevant_pages):

    relevant_count = sum(
        page in relevant_pages
        for page in retrieved_pages
    )

    return relevant_count / len(relevant_pages)

In [12]:
retrieved = [2,2,2,8,12]
relevant = [2]

recall = recall_at_k(retrieved, relevant)

print("Recall@5:", recall)

Recall@5: 3.0


In [16]:
def precision_at_k(retrieved_pages, relevant_pages):
    retrieved = set(retrieved_pages)
    relevant = set(relevant_pages)

    relevant_retrieved = retrieved.intersection(relevant)

    return len(relevant_retrieved) / len(retrieved_pages)


def recall_at_k(retrieved_pages, relevant_pages):
    retrieved = set(retrieved_pages)
    relevant = set(relevant_pages)

    relevant_retrieved = retrieved.intersection(relevant)

    return len(relevant_retrieved) / len(relevant)

In [19]:
import pandas as pd

df_results = pd.DataFrame(results)

df_results

,question,retrieved_pages,relevant_pages,precision@5,recall@5
0,What expenses are covered under inpatient hosp...,"[10, 2, 1, 2, 2]","[2, 3]",0.6,1.5
1,What is the pre-hospitalization period?,"[2, 2, 2, 8, 12]",[10],0.0,0.0
2,What is the post-hospitalization period?,"[2, 3, 8, 3, 10]",[10],0.2,1.0


In [20]:
results = []

evaluation_data = [
    {
        "question": "What expenses are covered under inpatient hospitalization?",
        "relevant_pages": [2, 3]
    },
    {
        "question": "What is the pre-hospitalization period?",
        "relevant_pages": [10]
    },
    {
        "question": "What is the post-hospitalization period?",
        "relevant_pages": [10]
    }
]

for item in evaluation_data:

    question = item["question"]
    relevant_pages = item["relevant_pages"]

    retrieved_pages = get_retrieved_pages(question, k=5)

    precision = precision_at_k(
        retrieved_pages,
        relevant_pages
    )

    recall = recall_at_k(
        retrieved_pages,
        relevant_pages
    )

    results.append({
        "question": question,
        "retrieved_pages": retrieved_pages,
        "relevant_pages": relevant_pages,
        "precision@5": precision,
        "recall@5": recall
    })

import pandas as pd

df_results = pd.DataFrame(results)

df_results

,question,retrieved_pages,relevant_pages,precision@5,recall@5
0,What expenses are covered under inpatient hosp...,"[10, 2, 1, 2, 2]","[2, 3]",0.2,0.5
1,What is the pre-hospitalization period?,"[2, 2, 2, 8, 12]",[10],0.0,0.0
2,What is the post-hospitalization period?,"[2, 3, 8, 3, 10]",[10],0.2,1.0


In [21]:
!pip install -U deepeval

In [24]:
from pathlib import Path

PDF_FOLDER = Path(".")

pdf_paths = list(PDF_FOLDER.glob("*.pdf"))

print("Total PDFs:", len(pdf_paths))

for p in pdf_paths:
    print(p.name)

Total PDFs: 7
care--policy-usage-guide.pdf
care-freedom-(health-insurance-product)---brochure.pdf
care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf
care-opd---prospectus-cum-sales-literature.pdf
care-plus-(health-insurance-product)---brochure.pdf
care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf
care-supreme----policy-usage-guide.pdf


In [26]:
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response["message"]["content"])

Hello!


In [27]:
from deepeval.models import OllamaModel

local_model = OllamaModel(
    model="llama3.2",
    base_url="http://localhost:11434"
)

print("Ollama model configured!")

Ollama model configured!


In [28]:
from deepeval.synthesizer import Synthesizer

synthesizer = Synthesizer(
    model=local_model
)

print("Synthesizer ready!")

Synthesizer ready!


In [30]:
from deepeval.models import OllamaEmbeddingModel

embedding_model = OllamaEmbeddingModel(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

print("Embedding model ready!")

Embedding model ready!


In [31]:
import ollama

print(ollama.list())

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 8, 25, 19, 11, 45, 940734, tzinfo=TzInfo(19800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]


In [32]:
import ollama

print(ollama.list())

models=[Model(model='nomic-embed-text:latest', modified_at=datetime.datetime(2026, 9, 26, 11, 18, 20, 638397, tzinfo=TzInfo(19800)), digest='0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', size=274302450, details=ModelDetails(parent_model='', format='gguf', family='nomic-bert', families=['nomic-bert'], parameter_size='137M', quantization_level='F16')), Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 8, 25, 19, 11, 45, 940734, tzinfo=TzInfo(19800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]


In [33]:
from deepeval.models import OllamaEmbeddingModel

embedding_model = OllamaEmbeddingModel(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

print("Embedding model ready!")

Embedding model ready!


In [36]:
import deepeval
import inspect
from deepeval.synthesizer import Synthesizer

print("DeepEval version:", deepeval.__version__)
print(inspect.signature(Synthesizer))

DeepEval version: 4.2.6
(model: Union[str, deepeval.models.base_model.DeepEvalBaseLLM, NoneType] = None, async_mode: bool = True, max_concurrent: int = 100, filtration_config: Optional[deepeval.synthesizer.config.FiltrationConfig] = None, evolution_config: Optional[deepeval.synthesizer.config.EvolutionConfig] = None, styling_config: Optional[deepeval.synthesizer.config.StylingConfig] = None, conversational_styling_config: Optional[deepeval.synthesizer.config.ConversationalStylingConfig] = None, cost_tracking: bool = False)


In [37]:
from deepeval.synthesizer import Synthesizer

synthesizer = Synthesizer(
    model=local_model,
    async_mode=False,
    max_concurrent=1
)

print("Synthesizer ready!")

Synthesizer ready!


In [38]:
import inspect

print(inspect.signature(synthesizer.generate_goldens_from_docs))

(document_paths: List[str], include_expected_output: bool = True, max_goldens_per_context: int = 2, context_construction_config: Optional[deepeval.synthesizer.config.ContextConstructionConfig] = None, _send_data=True) -> List[deepeval.dataset.golden.Golden]


In [41]:
from typing import List
import ollama

from deepeval.models import DeepEvalBaseEmbeddingModel


class OllamaEmbeddingModel(DeepEvalBaseEmbeddingModel):

    def __init__(self, model="nomic-embed-text"):
        self.model = model

    def embed_text(self, text: str) -> List[float]:
        response = ollama.embeddings(
            model=self.model,
            prompt=text
        )
        return response["embedding"]

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        return [self.embed_text(text) for text in texts]

    async def a_embed_text(self, text: str) -> List[float]:
        return self.embed_text(text)

    async def a_embed_texts(self, texts: List[str]) -> List[List[float]]:
        return self.embed_texts(texts)

    def get_model_name(self):
        return self.model

In [2]:
# Existing ChromaDB collection
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_collection(
    name="insurance_documents"
)

print("Total chunks:", collection.count())

Total chunks: 872


In [3]:
data = collection.get(
    include=["documents", "metadatas"]
)

documents = data["documents"]

print("Chunks loaded:", len(documents))

Chunks loaded: 872


In [4]:
contexts = []

context_size = 3

for i in range(0, len(documents), context_size):
    context = documents[i:i + context_size]

    if len(context) > 0:
        contexts.append(context)

print("Total contexts:", len(contexts))

Total contexts: 291


In [6]:
from deepeval.models import OllamaModel

local_model = OllamaModel(
    model="llama3.2",
    base_url="http://localhost:11434"
)

print("Local model ready!")

Local model ready!


In [7]:
from deepeval.synthesizer import Synthesizer

synthesizer = Synthesizer(
    model=local_model,
    async_mode=False,
    max_concurrent=1
)

print("Synthesizer ready!")

Synthesizer ready!


In [9]:
print("Number of contexts:", len(contexts))

print("First context:")
print(contexts[0] if contexts else "NO CONTEXTS FOUND")

Number of contexts: 291
First context:
["Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than 24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer", 'We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer Chemotherapy, Haemodialysis etc. Pre & Post-Hospitalization Medical Expenses Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics and prescribed medicines 30 days before date of admission and 60 days after date of discharge. Domiciliary Hospitalization Medical expenses incurred during his/her treatment at home so long as it invol

In [10]:
test_contexts = [contexts[0]]

print("Testing with:", len(test_contexts), "context")
print(test_contexts[0])

Testing with: 1 context
["Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than 24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer", 'We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer Chemotherapy, Haemodialysis etc. Pre & Post-Hospitalization Medical Expenses Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics and prescribed medicines 30 days before date of admission and 60 days after date of discharge. Domiciliary Hospitalization Medical expenses incurred during his/her treatment at home so long as it involves medical tre

In [12]:
import ollama
import json

context = contexts[0]

prompt = f"""
You are creating an evaluation dataset for an insurance RAG chatbot.

Read the following insurance policy context.

CONTEXT:
{context}

Generate exactly ONE factual question that can be answered using ONLY this context.

Return ONLY valid JSON in this exact format:
{{
    "question": "your question",
    "expected_answer": "your answer"
}}
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": prompt}
    ],
    options={"temperature": 0}
)

print(response["message"]["content"])

{
    "question": "How many day care treatments are covered under the insurance policy?",
    "expected_answer": "540"


In [13]:
import ollama
import json
import pandas as pd
import time

generated_data = []

for i, context in enumerate(contexts[:50]):

    prompt = f"""
You are creating an evaluation dataset for an insurance RAG chatbot.

Read the following insurance policy context.

CONTEXT:
{context}

Generate exactly ONE factual question that can be answered using ONLY this context.

Return ONLY valid JSON in this exact format:
{{
    "question": "your question",
    "expected_answer": "your answer"
}}
"""

    try:
        response = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "user", "content": prompt}
            ],
            options={"temperature": 0}
        )

        output = response["message"]["content"].strip()

        # Remove markdown code fences if model adds them
        output = output.replace("```json", "").replace("```", "").strip()

        result = json.loads(output)

        generated_data.append({
            "id": i + 1,
            "question": result["question"],
            "expected_answer": result["expected_answer"],
            "context": "\n\n".join(context)
        })

        print(f"Generated {i + 1}/50")

    except Exception as e:
        print(f"Error at {i + 1}: {e}")

    time.sleep(0.5)

df_eval = pd.DataFrame(generated_data)

print("\nTotal generated:", len(df_eval))

Error at 1: Expecting ',' delimiter: line 3 column 29 (char 118)
Error at 2: Expecting ',' delimiter: line 3 column 36 (char 134)
Error at 3: Expecting ',' delimiter: line 3 column 27 (char 133)
Error at 4: Expecting ',' delimiter: line 3 column 31 (char 184)
Error at 5: Expecting ',' delimiter: line 3 column 29 (char 135)
Generated 6/50
Error at 7: Expecting ',' delimiter: line 3 column 33 (char 117)
Generated 8/50
Error at 9: Expecting ',' delimiter: line 3 column 46 (char 165)
Error at 10: Expecting ',' delimiter: line 3 column 27 (char 180)
Error at 11: Expecting ',' delimiter: line 3 column 32 (char 163)
Generated 12/50
Generated 13/50
Error at 14: Expecting ',' delimiter: line 3 column 31 (char 109)
Generated 15/50
Error at 16: Expecting ',' delimiter: line 3 column 28 (char 164)
Error at 17: Expecting ',' delimiter: line 3 column 43 (char 142)
Error at 18: Expecting ',' delimiter: line 3 column 34 (char 134)
Generated 19/50
Error at 20: Expecting ',' delimiter: line 3 column 29 

In [14]:
df_eval.head(10)

,id,question,expected_answer,context
0,6,What is the required notice period for unplann...,48 hours,"of our empaneled network list , where in they ..."
1,8,How many years is the wait period for pre-exis...,2,"Health Insurance for Diabetes, High Blood Pres..."
2,12,What is the duration of the free-look period f...,30 days,more! Avail a discount on your premium if you ...
3,13,How many days of home care can be covered per ...,7,policy year if the insured person is admitted ...
4,15,At what age does an Insured Person (or eldest ...,71,the Insured Person (or eldest Insured Person i...
5,19,What type of treatments are covered only after...,"non-infective arthritis, cataract, joint repla...",9 th Annual India Insurance Summit & Awards 20...
6,25,How many consecutive days of hospitalization e...,10,to a Hospital for treatment of any One Illness...
7,26,What is the minimum duration of hospitalizatio...,3 consecutive days,- perhaps Your state of health is such that Yo...
8,27,What is the maximum claim amount payable for a...,The Sum Insured,"You ever run out of/ exhaust Your Sum Insured,..."
9,29,At what age does an Insured Person (or eldest ...,71,i. You will bear a Co-payment of 20% / 30% per...


In [15]:
df_eval.to_csv(
    "insurance_evaluation_dataset_generated.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [16]:
df_eval.head(10)

,id,question,expected_answer,context
0,6,What is the required notice period for unplann...,48 hours,"of our empaneled network list , where in they ..."
1,8,How many years is the wait period for pre-exis...,2,"Health Insurance for Diabetes, High Blood Pres..."
2,12,What is the duration of the free-look period f...,30 days,more! Avail a discount on your premium if you ...
3,13,How many days of home care can be covered per ...,7,policy year if the insured person is admitted ...
4,15,At what age does an Insured Person (or eldest ...,71,the Insured Person (or eldest Insured Person i...
5,19,What type of treatments are covered only after...,"non-infective arthritis, cataract, joint repla...",9 th Annual India Insurance Summit & Awards 20...
6,25,How many consecutive days of hospitalization e...,10,to a Hospital for treatment of any One Illness...
7,26,What is the minimum duration of hospitalizatio...,3 consecutive days,- perhaps Your state of health is such that Yo...
8,27,What is the maximum claim amount payable for a...,The Sum Insured,"You ever run out of/ exhaust Your Sum Insured,..."
9,29,At what age does an Insured Person (or eldest ...,71,i. You will bear a Co-payment of 20% / 30% per...


In [17]:
print("Total questions:", len(df_eval))

Total questions: 21


In [18]:
pd.set_option("display.max_colwidth", 200)

df_eval[["id", "question", "expected_answer"]]

,id,question,expected_answer
0,6,What is the required notice period for unplanned hospitalization?,48 hours
1,8,How many years is the wait period for pre-existing diseases under this health insurance plan?,2
2,12,What is the duration of the free-look period for the insurance policy?,30 days
3,13,How many days of home care can be covered per illness/injury under the policy?,7
4,15,At what age does an Insured Person (or eldest Insured Person in the case of a Floater cover) face an additional 10% co-payment for the Policy?,71
5,19,What type of treatments are covered only after completion of 2 consecutive policy years?,"non-infective arthritis, cataract, joint replacement"
6,25,How many consecutive days of hospitalization exceeds the policy period before the insurance company will not be liable to make payment under the policy?,10
7,26,What is the minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit?,3 consecutive days
8,27,What is the maximum claim amount payable for a single claim during a Policy Year?,The Sum Insured
9,29,At what age does an Insured Person (or eldest Insured Person in the case of a Floater cover) trigger an additional 10% co-payment?,71


In [19]:
for i, row in df_eval.iterrows():
    print(f"\n{i+1}. Question: {row['question']}")
    print(f"   Answer: {row['expected_answer']}")


1. Question: What is the required notice period for unplanned hospitalization?
   Answer: 48 hours

2. Question: How many years is the wait period for pre-existing diseases under this health insurance plan?
   Answer: 2

3. Question: What is the duration of the free-look period for the insurance policy?
   Answer: 30 days

4. Question: How many days of home care can be covered per illness/injury under the policy?
   Answer: 7

5. Question: At what age does an Insured Person (or eldest Insured Person in the case of a Floater cover) face an additional 10% co-payment for the Policy?
   Answer: 71

6. Question: What type of treatments are covered only after completion of 2 consecutive policy years?
   Answer: non-infective arthritis, cataract, joint replacement

7. Question: How many consecutive days of hospitalization exceeds the policy period before the insurance company will not be liable to make payment under the policy?
   Answer: 10

8. Question: What is the minimum duration of hosp

In [20]:
for i, row in df_eval.iterrows():
    print(f"\n{'='*80}")
    print(f"ID: {row['id']}")
    print(f"QUESTION: {row['question']}")
    print(f"ANSWER: {row['expected_answer']}")
    print(f"CONTEXT:\n{row['context'][:1500]}")


ID: 6
QUESTION: What is the required notice period for unplanned hospitalization?
ANSWER: 48 hours
CONTEXT:
of our empaneled network list , where in they can settle the hospitalization bill and claim for the reimbursements of same as per policy terms & conditions, after getting discharged. • We request all our insured members to intimate us of any accident or illness (for cashless claims) before hospitalization. For unplanned hospitalization within 48 hours of admission and in case of planned hospitalization at least 48 hours prior to the planned date of admission to hospital. • For hassle free claim intimation

48 hours prior to the planned date of admission to hospital. • For hassle free claim intimation : just scan and upload the claim documents at self-help portal link. • For hard copy claim intimation send us at : Vipul Tech Square, Tower C, 3 rd Floor, Golf Course Road, Sector-43, Gurugram-122009 (Haryana) HOW TO CLAIM

Care Health Insurance Limited Regd. Office: 5 th Floor, 19 

In [21]:
# Create cleaned evaluation dataset

remove_ids = [10, 12, 15, 19]

df_clean = df_eval[~df_eval["id"].isin(remove_ids)].copy()

# Correct question #7
df_clean.loc[df_clean["id"] == 25, "question"] = (
    "After how many consecutive days of hospitalization is the insurer "
    "not liable to make payment under this benefit?"
)
df_clean.loc[df_clean["id"] == 25, "expected_answer"] = "10 days"

# Correct question #20
df_clean.loc[df_clean["id"] == 49, "question"] = (
    "What is the specific waiting period for listed conditions, "
    "surgeries, or treatments?"
)
df_clean.loc[df_clean["id"] == 49, "expected_answer"] = (
    "24 months of continuous coverage"
)

print("Cleaned questions:", len(df_clean))

Cleaned questions: 18


In [22]:
df_clean[["id", "question", "expected_answer"]]

,id,question,expected_answer
0,6,What is the required notice period for unplanned hospitalization?,48 hours
1,8,How many years is the wait period for pre-existing diseases under this health insurance plan?,2
3,13,How many days of home care can be covered per illness/injury under the policy?,7
6,25,After how many consecutive days of hospitalization is the insurer not liable to make payment under this benefit?,10 days
7,26,What is the minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit?,3 consecutive days
8,27,What is the maximum claim amount payable for a single claim during a Policy Year?,The Sum Insured
9,29,At what age does an Insured Person (or eldest Insured Person in the case of a Floater cover) trigger an additional 10% co-payment?,71
10,30,How many regular doctor consultations are covered under the Good Health+ Optional Cover?,8
11,31,How many consecutive days of medical expenses are covered under the policy?,7
12,33,What is the duration of the Free Look Period for new individual health insurance policies?,30 days


In [23]:
df_clean.to_csv(
    "insurance_evaluation_dataset_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [24]:
import chromadb
from sentence_transformers import SentenceTransformer

# ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="insurance_documents")

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Chunks in database:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks in database: 872


In [25]:
def retrieve_contexts(question, top_k=5):
    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]

In [26]:
question = df_clean.iloc[0]["question"]

retrieved = retrieve_contexts(question, top_k=5)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXTS:")
for i, ctx in enumerate(retrieved, 1):
    print(f"\n--- Context {i} ---")
    print(ctx[:1000])

QUESTION:
What is the required notice period for unplanned hospitalization?

RETRIEVED CONTEXTS:

--- Context 1 ---
within 15 days from the date of discharge from the Hospital. Note: The above points ((i) & (ii)) are precedent to admission of liability under the policy. (iii) In case of an Emergency Hospitalization, We shall be notified either at the Our call center or in writing immediately and in any event within 48 hours of Hospitalization commencing or before the Insured Person’s discharge from Hospital. However, we will examine & relax the time limit mentioned in the above conditions depending upon the merits of

--- Context 2 ---
immediately before Insured Person’s Hospitalization valid from the Policy Start Date; and (ii) The Medical Expenses incurred by You immediately after Insured Person’s discharge from Hospital valid till 30 days beyond the Policy End Date. Provided that the Medical Expenses relate to the Illness/ Injury for which We have accepted the Insured Person’s Claim

In [27]:
import ollama
import pandas as pd

evaluation_data = []

for _, row in df_clean.iterrows():

    question = row["question"]
    reference = row["expected_answer"]

    # Retrieve top 5 contexts
    retrieved = retrieve_contexts(question, top_k=5)

    # Combine contexts for LLM
    context_text = "\n\n".join(retrieved)

    prompt = f"""
You are an insurance RAG chatbot.

Answer the user's question using ONLY the provided context.
Do not use outside knowledge.
If the answer is not present in the context, say that it is not available.

CONTEXT:
{context_text}

QUESTION:
{question}

Give a concise factual answer.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0}
    )

    answer = response["message"]["content"].strip()

    evaluation_data.append({
        "user_input": question,
        "retrieved_contexts": retrieved,
        "response": answer,
        "reference": reference
    })

    print(f"Completed: {len(evaluation_data)}/{len(df_clean)}")

Completed: 1/18
Completed: 2/18
Completed: 3/18
Completed: 4/18
Completed: 5/18
Completed: 6/18
Completed: 7/18
Completed: 8/18
Completed: 9/18
Completed: 10/18
Completed: 11/18
Completed: 12/18
Completed: 13/18
Completed: 14/18
Completed: 15/18
Completed: 16/18
Completed: 17/18
Completed: 18/18


In [28]:
df_ragas = pd.DataFrame(evaluation_data)

print("Total evaluation samples:", len(df_ragas))
df_ragas[["user_input", "response", "reference"]]

Total evaluation samples: 18


,user_input,response,reference
0,What is the required notice period for unplanned hospitalization?,The required notice period for unplanned hospitalization is 24 hours.,48 hours
1,How many years is the wait period for pre-existing diseases under this health insurance plan?,The wait period for pre-existing diseases under this health insurance plan is 24 months.,2
2,How many days of home care can be covered per illness/injury under the policy?,Maximum of 7 days per illness/injury.,7
3,After how many consecutive days of hospitalization is the insurer not liable to make payment under this benefit?,The insurer is not liable to make payment under this benefit for the first 3 consecutive days of hospitalization.,10 days
4,What is the minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit?,The minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit is 3 consecutive days.,3 consecutive days
5,What is the maximum claim amount payable for a single claim during a Policy Year?,"The maximum claim amount payable for a single claim during a Policy Year is the sum total of Sum Insured, Cumulative Bonus, and Inflation Shield.",The Sum Insured
6,At what age does an Insured Person (or eldest Insured Person in the case of a Floater cover) trigger an additional 10% co-payment?,An Insured Person (or eldest Insured Person in the case of a Floater cover) triggers an additional 10% co-payment at the age of 71.,71
7,How many regular doctor consultations are covered under the Good Health+ Optional Cover?,Up to 8 consultations are covered under the Good Health+ Optional Cover.,8
8,How many consecutive days of medical expenses are covered under the policy?,"The policy covers up to 7 consecutive days of medical expenses for expenses arising from Any One Illness or an Injury, and up to 45 days for the first day of hiring a Qualified Nurse.",7
9,What is the duration of the Free Look Period for new individual health insurance policies?,The duration of the Free Look Period for new individual health insurance policies is 30 days.,30 days


In [29]:
df_ragas.to_pickle("insurance_ragas_evaluation_data.pkl")

print("RAGAS evaluation data saved successfully!")

RAGAS evaluation data saved successfully!


In [30]:
import ragas
print("RAGAS version:", ragas.__version__)

from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

print("RAGAS imports successful!")

RAGAS version: 0.4.3
RAGAS imports successful!


C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\3523023020.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\3523023020.py:5: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\3523023020.py:5: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Loc

In [31]:
from ragas import EvaluationDataset, SingleTurnSample

samples = []

for _, row in df_ragas.iterrows():
    samples.append(
        SingleTurnSample(
            user_input=row["user_input"],
            retrieved_contexts=row["retrieved_contexts"],
            response=row["response"],
            reference=row["reference"]
        )
    )

ragas_dataset = EvaluationDataset(samples=samples)

print("RAGAS samples:", len(ragas_dataset.samples))

RAGAS samples: 18


In [32]:
print("df_clean:", len(df_clean))
print("df_ragas:", len(df_ragas))
print("RAGAS samples:", len(ragas_dataset.samples))

df_clean: 18
df_ragas: 18
RAGAS samples: 18


In [33]:
df_ragas[["user_input", "response", "reference"]]

,user_input,response,reference
0,What is the required notice period for unplanned hospitalization?,The required notice period for unplanned hospitalization is 24 hours.,48 hours
1,How many years is the wait period for pre-existing diseases under this health insurance plan?,The wait period for pre-existing diseases under this health insurance plan is 24 months.,2
2,How many days of home care can be covered per illness/injury under the policy?,Maximum of 7 days per illness/injury.,7
3,After how many consecutive days of hospitalization is the insurer not liable to make payment under this benefit?,The insurer is not liable to make payment under this benefit for the first 3 consecutive days of hospitalization.,10 days
4,What is the minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit?,The minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit is 3 consecutive days.,3 consecutive days
5,What is the maximum claim amount payable for a single claim during a Policy Year?,"The maximum claim amount payable for a single claim during a Policy Year is the sum total of Sum Insured, Cumulative Bonus, and Inflation Shield.",The Sum Insured
6,At what age does an Insured Person (or eldest Insured Person in the case of a Floater cover) trigger an additional 10% co-payment?,An Insured Person (or eldest Insured Person in the case of a Floater cover) triggers an additional 10% co-payment at the age of 71.,71
7,How many regular doctor consultations are covered under the Good Health+ Optional Cover?,Up to 8 consultations are covered under the Good Health+ Optional Cover.,8
8,How many consecutive days of medical expenses are covered under the policy?,"The policy covers up to 7 consecutive days of medical expenses for expenses arising from Any One Illness or an Injury, and up to 45 days for the first day of hiring a Qualified Nurse.",7
9,What is the duration of the Free Look Period for new individual health insurance policies?,The duration of the Free Look Period for new individual health insurance policies is 30 days.,30 days


In [34]:
from ragas.llms import LangchainLLMWrapper
from langchain_ollama import ChatOllama

evaluator_llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

evaluator_llm = LangchainLLMWrapper(evaluator_llm)

print("RAGAS evaluator LLM ready!")

RAGAS evaluator LLM ready!


C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\359359307.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(evaluator_llm)


In [35]:
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    ContextPrecision(),
    ContextRecall()
]

print("Metrics ready:")
for metric in metrics:
    print("-", metric.name)

Metrics ready:
- faithfulness
- answer_relevancy
- context_precision
- context_recall


C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\337519960.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\337519960.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_22624\337519960.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\LENOVO 4HIN\AppData\Local\

In [37]:
print(type(evaluator_llm))
print(type(metrics[0]))

<class 'ragas.llms.base.LangchainLLMWrapper'>
<class 'ragas.metrics._faithfulness.Faithfulness'>


In [38]:
for metric in metrics:
    print(metric.name)
    print("LLM:", getattr(metric, "llm", None))
    print("Embeddings:", getattr(metric, "embeddings", None))

faithfulness
LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Embeddings: None
answer_relevancy
LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Embeddings: None
context_precision
LLM: None
Embeddings: None
context_recall
LLM: None
Embeddings: None


In [39]:
metrics[2].llm = evaluator_llm
metrics[3].llm = evaluator_llm

for metric in metrics:
    print(metric.name, "->", metric.llm)

faithfulness -> LangchainLLMWrapper(langchain_llm=ChatOllama(...))
answer_relevancy -> LangchainLLMWrapper(langchain_llm=ChatOllama(...))
context_precision -> LangchainLLMWrapper(langchain_llm=ChatOllama(...))
context_recall -> LangchainLLMWrapper(langchain_llm=ChatOllama(...))


In [44]:
for metric in metrics:
    print("\nMETRIC:", metric.name)
    print("LLM:", metric.llm)
    print("Embeddings:", getattr(metric, "embeddings", None))


METRIC: faithfulness
LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Embeddings: None

METRIC: answer_relevancy
LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Embeddings: None

METRIC: context_precision
LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Embeddings: None

METRIC: context_recall
LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Embeddings: None


In [46]:
from langchain_ollama import OllamaEmbeddings

evaluator_embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

print("Ollama embeddings ready!")

Ollama embeddings ready!


In [47]:
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    embeddings=evaluator_embeddings
)

print(result)

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[9]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[19]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[21]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[24]: TimeoutError

{'faithfulness': nan, 'answer_relevancy': 0.9265, 'context_precision': nan, 'context_recall': 0.7630}


In [48]:
print(result.to_pandas())

                                                                                                                                             user_input  \
0                                                                                     What is the required notice period for unplanned hospitalization?   
1                                                         How many years is the wait period for pre-existing diseases under this health insurance plan?   
2                                                                        How many days of home care can be covered per illness/injury under the policy?   
3                                      After how many consecutive days of hospitalization is the insurer not liable to make payment under this benefit?   
4   What is the minimum duration of hospitalization required for the insurance to cover medical expenses under the Domiciliary Hospitalization Benefit?   
5                                                                     

In [49]:
print(result)

{'faithfulness': nan, 'answer_relevancy': 0.9265, 'context_precision': nan, 'context_recall': 0.7630}


In [50]:
result_df = result.to_pandas()

print(result_df[[
    "user_input",
    "faithfulness",
    "context_precision",
    "answer_relevancy",
    "context_recall"
]].to_string(index=False))

                                                                                                                                         user_input  faithfulness  context_precision  answer_relevancy  context_recall
                                                                                  What is the required notice period for unplanned hospitalization?           NaN                NaN               NaN        0.416667
                                                      How many years is the wait period for pre-existing diseases under this health insurance plan?           NaN                NaN               NaN             NaN
                                                                     How many days of home care can be covered per illness/injury under the policy?           NaN                NaN               NaN             NaN
                                   After how many consecutive days of hospitalization is the insurer not liable to make payment under this b

In [51]:
print(result)

{'faithfulness': nan, 'answer_relevancy': 0.9265, 'context_precision': nan, 'context_recall': 0.7630}


In [52]:
ragas_summary = pd.DataFrame({
    "Metric": [
        "Answer Relevancy",
        "Context Recall",
        "Faithfulness",
        "Context Precision"
    ],
    "Score": [
        0.9265,
        0.7630,
        None,
        None
    ]
})

ragas_summary

,Metric,Score
0,Answer Relevancy,0.9265
1,Context Recall,0.7630
2,Faithfulness,NaN
3,Context Precision,NaN


In [54]:
print(ragas_dataset.samples[0])

user_input='What is the required notice period for unplanned hospitalization?' retrieved_contexts=['within 15 days from the date of discharge from the Hospital. Note: The above points ((i) & (ii)) are precedent to admission of liability under the policy. (iii) In case of an Emergency Hospitalization, We shall be notified either at the Our call center or in writing immediately and in any event within 48 hours of Hospitalization commencing or before the Insured Person’s discharge from Hospital. However, we will examine & relax the time limit mentioned in the above conditions depending upon the merits of', 'immediately before Insured Person’s Hospitalization valid from the Policy Start Date; and (ii) The Medical Expenses incurred by You immediately after Insured Person’s discharge from Hospital valid till 30 days beyond the Policy End Date. Provided that the Medical Expenses relate to the Illness/ Injury for which We have accepted the Insured Person’s Claim.', 'for a period of 90 days imm

In [55]:
sample = ragas_dataset.samples[0]

print("QUESTION:")
print(sample.user_input)

print("\nRESPONSE:")
print(sample.response)

print("\nCONTEXTS:")
for i, ctx in enumerate(sample.retrieved_contexts, 1):
    print(f"\n--- Context {i} ---")
    print(ctx[:500])

QUESTION:
What is the required notice period for unplanned hospitalization?

RESPONSE:
The required notice period for unplanned hospitalization is 24 hours.

CONTEXTS:

--- Context 1 ---
within 15 days from the date of discharge from the Hospital. Note: The above points ((i) & (ii)) are precedent to admission of liability under the policy. (iii) In case of an Emergency Hospitalization, We shall be notified either at the Our call center or in writing immediately and in any event within 48 hours of Hospitalization commencing or before the Insured Person’s discharge from Hospital. However, we will examine & relax the time limit mentioned in the above conditions depending upon the m

--- Context 2 ---
immediately before Insured Person’s Hospitalization valid from the Policy Start Date; and (ii) The Medical Expenses incurred by You immediately after Insured Person’s discharge from Hospital valid till 30 days beyond the Policy End Date. Provided that the Medical Expenses relate to the Illnes

In [56]:
faithfulness_metric = Faithfulness(llm=evaluator_llm)
context_precision_metric = ContextPrecision(llm=evaluator_llm)

print("Faithfulness LLM:", faithfulness_metric.llm)
print("Context Precision LLM:", context_precision_metric.llm)

Faithfulness LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))
Context Precision LLM: LangchainLLMWrapper(langchain_llm=ChatOllama(...))


In [57]:
test_metrics = [
    faithfulness_metric,
    context_precision_metric
]

test_result = evaluate(
    dataset=EvaluationDataset(
        samples=[ragas_dataset.samples[0]]
    ),
    metrics=test_metrics,
    embeddings=evaluator_embeddings
)

print(test_result)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Exception raised in Job[1]: OutputParserException(Invalid json output: The provided context was not useful in arriving at the given answer. The context appears to be related to health insurance policies and does not provide any information about the required notice period for unplanned hospitalization.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


{'faithfulness': 1.0000, 'context_precision': nan}


In [59]:
from ragas import RunConfig

run_config = RunConfig(
    timeout=180,
    max_retries=2
)

print(run_config)

RunConfig(timeout=180, max_retries=2, max_wait=60, max_workers=16, exception_types=(<class 'Exception'>,), log_tenacity=False, seed=42)
